In [ ]:

# Scenario: Using LLM-based Query Decomposition and Query Routing to 
# retrieve complementary information from separate fruit and nutrition knowledge bases 
# for a unified RAG answer.

# ============================================================
# 1. LOAD DOCUMENTS
# ============================================================

from langchain_community.document_loaders import TextLoader

loader_fruits = TextLoader(
    "./notes/fruits_and_vegetables.txt",
    encoding="utf-8"
)

loader_nutrition = TextLoader(
    "./notes/human_nutrition_and_dietary_requirements.txt",
    encoding="utf-8"
)

documents_fruits = loader_fruits.load()
documents_nutrition = loader_nutrition.load()


# ============================================================
# 2. SPLIT DOCUMENTS INTO CHUNKS
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks_fruits = text_splitter.split_documents(documents_fruits)
chunks_nutrition = text_splitter.split_documents(documents_nutrition)


# ============================================================
# 3. GENERATE EMBEDDINGS
# ============================================================

from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)


# ============================================================
# 4. STORE CHUNKS IN TWO FAISS VECTOR STORES
# ============================================================

from langchain_community.vectorstores import FAISS

vectorstore_fruits = FAISS.from_documents(
    chunks_fruits,
    embedding_model
)

vectorstore_nutrition = FAISS.from_documents(
    chunks_nutrition,
    embedding_model
)


# ============================================================
# 5. CREATE TWO RETRIEVERS
# ============================================================

fruits_retriever = vectorstore_fruits.as_retriever(
    search_kwargs={"k": 3}
)

nutrition_retriever = vectorstore_nutrition.as_retriever(
    search_kwargs={"k": 3}
)


# ============================================================
# 6. INITIALIZE THE LLM
# ============================================================

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0
)


# ============================================================
# 7. ACCEPT THE USER QUERY
# ============================================================

user_query = (
    "Are bananas good for someone who needs potassium, "
    "and what role does potassium play in the human body?"
)


# ============================================================
# 8. DYNAMIC QUERY DECOMPOSITION USING AN LLM
# ============================================================

from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate

# Define the expected structured output

class QueryDecomposition(BaseModel):
    fruit_query: str
    nutrition_query: str


# Create the decomposition prompt

decomposition_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        Break the user's question into two targeted queries.

        1. fruit_query: Ask about the relevant fruit and its
           nutritional properties.

        2. nutrition_query: Ask about the nutrient and its
           functions in the human body.

        Return only the two required fields in the
        specified structured format.
        """
    ),
    ("human", "{question}")
])


# Configure the LLM to return structured output

decomposition_llm = llm.with_structured_output(
    QueryDecomposition,
    method="json_schema"
)


# Build the decomposition chain

decomposition_chain = (
    decomposition_prompt
    | decomposition_llm
)


# Generate subqueries dynamically

queries = decomposition_chain.invoke({
    "question": user_query
})

fruit_query = queries.fruit_query
nutrition_query = queries.nutrition_query

print("Fruit Query:", fruit_query)
print("Nutrition Query:", nutrition_query)


# ============================================================
# 9. QUERY ROUTING AND RETRIEVAL
# ============================================================

# Route each generated query to its relevant retriever

fruit_docs = fruits_retriever.invoke(fruit_query)

nutrition_docs = nutrition_retriever.invoke(nutrition_query)


# ============================================================
# 10. COMBINE RETRIEVED DOCUMENTS
# ============================================================

all_docs = fruit_docs + nutrition_docs

# Remove duplicate chunks

unique_docs = []
seen = set()

for doc in all_docs:
    if doc.page_content not in seen:
        unique_docs.append(doc)
        seen.add(doc.page_content)


# ============================================================
# 11. BUILD THE AUGMENTED CONTEXT
# ============================================================

context = "\n\n".join(
    doc.page_content for doc in unique_docs
)


# ============================================================
# 12. CREATE THE FINAL RAG PROMPT
# ============================================================

from langchain_core.output_parsers import StrOutputParser

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a nutrition information assistant.

        Answer using only the supplied context.
        Explain the nutritional information and the role
        of nutrients in the human body.

        If the context does not contain sufficient information,
        clearly mention that. Do not invent nutritional values.
        """
    ),
    (
        "human",
        """
        Context:
        {context}

        User Question:
        {question}

        Provide a clear and accurate answer.
        """
    )
])


# ============================================================
# 13. GENERATE THE FINAL ANSWER
# ============================================================

rag_chain = (
    answer_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke({
    "context": context,
    "question": user_query
})

print("\nRetrieved Chunks:", len(unique_docs))

print("\nFinal Answer:")
print(answer)
